<a href="https://colab.research.google.com/github/anandharidas/KernelProgramming/blob/main/Type_4_Matrix_mutiplier_using_1D_Tiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
%%writefile matmul-04.py
from numba import cuda, float32
import cupy as cp
import numpy as np

BM = 32
BN = 32
BK = 32
TM = 8  # no. of output elements process in single thread
NT = (BM * BN) // TM

ceil_div = lambda a, b: (a + b - 1) // b

@cuda.jit
def matmul(matA, matB, matC, m, n, k):
  tileA = cuda.shared.array(shape=(BM, BK), dtype=float32)
  tileB = cuda.shared.array(shape=(BK, BN), dtype=float32)

  tid = cuda.threadIdx.x
  col = tid % BN
  row = (tid // BN) * TM

  block_row = cuda.blockIdx.y * BM
  block_col = cuda.blockIdx.x * BN

  # register allocation and initialization
  acc = cuda.local.array(TM, dtype=float32)
  for t in range(TM):
    acc[t] = float32(0.0)

  for k0 in range(0, k, BK):
    # phase 1: load values from matA into tileA
    for i in range(tid, BM * BK, NT):
      r = i // BK
      c = i % BK
      global_mem_row = block_row + r
      global_mem_col = k0 + c
      if global_mem_row < m and global_mem_col < k:
        tileA[r, c] = matA[global_mem_row, global_mem_col]
      else:
        tileA[r, c] = float32(0.0)

    # phase 1: load values from matB into tileB
    for i in range(tid, BK * BN, NT):
      r = i // BN
      c = i % BN
      global_mem_row = k0 + r
      global_mem_col = c + block_col
      if global_mem_row < k and global_mem_col < n:
        tileB[r, c] = matB[global_mem_row, global_mem_col]
      else:
        tileB[r, c] = float32(0.0)

    # sync threads
    cuda.syncthreads()

    # phase 2: compute accumulation
    for kk in range(BK):
      b = tileB[kk, col]
      for t in range(TM):
        acc[t] += tileA[row + t, kk] * b

    # sync threads
    cuda.syncthreads()

  global_mem_col = block_col + col
  for t in range(TM):
    global_mem_row = block_row + row + t
    if global_mem_row < m and global_mem_col < n:
      matC[global_mem_row, global_mem_col] = acc[t]


m = 1000
k = 1200
n = 800

matA = cp.ones(shape=(m, k), dtype=cp.float32) * 2.0
matB = cp.ones(shape=(k, n), dtype=cp.float32) * 3.0
matC = cp.zeros(shape=(m, n), dtype=cp.float32)

threads_per_block = (NT, 1)
blocks_per_grid = (ceil_div(n, BN), ceil_div(m, BM))

matmul[blocks_per_grid, threads_per_block](matA, matB, matC, m, n, k)

cuda.synchronize()

matA_cpu = cp.asnumpy(matA)
matB_cpu = cp.asnumpy(matB)
matC_cpu = cp.asnumpy(matC)

np.testing.assert_allclose(matC_cpu, matA_cpu @ matB_cpu, rtol=1e-5, atol=1e-5)
print("Assertion passed! Results match NumPy reference.")

Overwriting matmul-04.py


In [6]:
!ncu --set full -f -o matmul-04.ncu-rep python3 matmul-04.py

==PROF== Connected to process 2743 (/usr/bin/python3.13)
==PROF== Profiling "cupy_fill" - 0: 0%....50%....100% - 50 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 1: 0%....50%....100% - 50 passes
==PROF== Profiling "cupy_fill" - 2: 0%....50%....100% - 51 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 3: 0%....50%....100% - 51 passes
==PROF== Profiling "matmul[abi:v1,cw51cXTLSUwv1sD..." - 4: 0%....50%....100% - 50 passes
Assertion passed! Results match NumPy reference.
==PROF== Disconnected from process 2743
==PROF== Report: /content/matmul-04.ncu-rep
